# Scaled Dot-Product Attention with a Causal Mask

**Topic:** Attention Mechanisms &nbsp;·&nbsp; **Difficulty:** Hard

## The data (created in the next cell)

- `Q`, `K`, `V` are arrays of shape `(4, 10, 16)`: batch of 4 sequences, length 10, dimension 16.
- The attention must be **causal**: position *i* may attend only to positions *j ≤ i*.
- A reference implementation hides in the evaluation cell.

## Your task

With **NumPy only** (no torch):

1. Compute `scores = Q @ K.transpose(0, 2, 1) / sqrt(16)`.
2. Set masked (future) positions to `-inf` **before** the softmax, and use a numerically stable softmax (subtract the row max).
3. Set `attn_out` to the `(4, 10, 16)` result of `weights @ V`.

## Evaluation

Max absolute difference from the reference must be below 1e-6.


In [ ]:
import numpy as np

rng = np.random.default_rng(421)
B, N, D = 4, 10, 16
Q = rng.normal(0, 1.0, (B, N, D))
K = rng.normal(0, 1.0, (B, N, D))
V = rng.normal(0, 1.0, (B, N, D))
print(f"Q/K/V shape: {Q.shape}. Produce attn_out with the same shape (causal).")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: compare against an independent reference implementation
# =====================================================================
import numpy as np

if "attn_out" not in dir():
    raise NameError("Define `attn_out` in the solution cell before running this cell.")

_scores = Q @ K.transpose(0, 2, 1) / np.sqrt(Q.shape[-1])
_mask = np.tril(np.ones((Q.shape[1], Q.shape[1]), dtype=bool))
_scores = np.where(_mask[None], _scores, -np.inf)
_scores -= _scores.max(axis=-1, keepdims=True)
_w = np.exp(_scores)
_w /= _w.sum(axis=-1, keepdims=True)
_ref = _w @ V

_out = np.asarray(attn_out, dtype=float)
if _out.shape != _ref.shape:
    raise ValueError(f"`attn_out` has shape {_out.shape}, expected {_ref.shape}.")
_err = float(np.max(np.abs(_out - _ref)))
print("=" * 52)
print(f"Max absolute error vs reference: {_err:.2e}   (need < 1e-6)")
print(f"Row 0 attention sanity: position 0 must attend only to itself.")
print("=" * 52)
print("PASSED: that's the heart of a transformer!" if _err < 1e-6 else "NOT PASSED yet: check the mask, the scaling, and the softmax axis.")


## Solution

Scale the score matrix by 1/sqrt(d), overwrite the upper triangle (future positions) with -inf, then apply a max-subtracted softmax over the last axis and multiply by V.

```python
import numpy as np

d = Q.shape[-1]
scores = Q @ K.transpose(0, 2, 1) / np.sqrt(d)

mask = np.tril(np.ones((Q.shape[1], Q.shape[1]), dtype=bool))
scores = np.where(mask[None], scores, -np.inf)

scores -= scores.max(axis=-1, keepdims=True)   # numerical stability
weights = np.exp(scores)
weights /= weights.sum(axis=-1, keepdims=True)

attn_out = weights @ V
print(attn_out.shape)
```
